# 07 工程工具核心八股（Git / Linux / pytest / 设计模式 / 性能 / 进程管理）

> 覆盖范围：Git 命令与分支模型、Linux 排查命令、pytest 测试框架、常用设计模式的 Python 实现、性能分析与线上排查、进程管理与 Docker。
> 使用方式：命令/代码可直接复制运行；每节含**对比表**与**面试追问**，速记卡片见文末附录。

---

## 1. Git 版本控制

### 1.1 核心命令清单

| 命令 | 作用 | 典型用法 / 参数 |
| --- | --- | --- |
| `git init` | 初始化仓库 | `git init -b main` 指定初始分支 |
| `git add` | 暂存改动 | `git add -p` 分块暂存 |
| `git commit` | 提交快照 | `git commit -am "msg"` 跳过暂存直接提交已跟踪文件 |
| `git status` / `git diff` | 查看状态/差异 | `git diff --staged` 看已暂存差异 |
| `git branch` | 分支管理 | `git branch -d name` / `-D` 强制删 |
| `git checkout` / `switch` | 切换分支 | `git switch -c feature/xxx` 新建并切换 |
| `git merge` | 合并（保留历史） | `git merge --no-ff feature` |
| `git rebase` | 变基（重写历史） | `git rebase -i HEAD~3` 交互式整理提交 |
| `git cherry-pick` | 挑选提交到当前分支 | `git cherry-pick <commit-hash>` |
| `git stash` | 暂存未提交改动 | `git stash push -m "wip"` / `git stash pop` |
| `git reset` | 回退（**重写历史**） | `--soft`(留暂存) / `--mixed`(留工作区) / `--hard`(全丢) |
| `git revert` | 反向提交（**追加新提交**） | `git revert <commit-hash>` 安全回退已推送提交 |
| `git log` | 查看提交历史 | `git log --oneline --graph --all` |
| `git reflog` | 查看所有指针移动 | 找回被 reset 掉的提交 |
| `git fetch` / `pull` / `push` | 远程同步 | `git pull --rebase` 拉取并变基 |

### 1.2 分支模型：Git Flow vs GitHub Flow

**Git Flow（适合版本发布节奏固定、需要长期维护分支的团队）**：

```
master（稳定发布） ← merge --no-ff ← develop（集成分支）
develop ← feature/*（功能分支）
master ← hotfix/*（线上热修，直接合 master + develop）
master ← release/*（发布准备，只修 bug）
```

**GitHub Flow（适合持续部署、发布频繁的团队）**：

```
main（永远可部署） ← feature 分支 → PR（Code Review） → 合并后自动部署
```

| 维度 | Git Flow | GitHub Flow |
| --- | --- | --- |
| 分支数量 | 多（master/develop/feature/release/hotfix） | 少（main + 临时 feature） |
| 发布方式 | 版本节奏式（release 分支冻结） | 持续部署（PR 合入即上线） |
| 历史复杂度 | 复杂、`--no-ff` 合并图密集 | 线性、清爽 |
| 适用团队 | 传统版本发布、多版本并存维护 | 互联网快速迭代、CI/CD 成熟 |
| 学习成本 | 高 | 低 |

**选择要点**：需要同时维护 v1/v2/v3 多版本 → Git Flow；单版本快速上线 → GitHub Flow（或 trunk-based）。

### 1.3 冲突解决步骤

冲突发生在 merge / rebase / cherry-pick 时（双方修改同一文件同一区域）。标准流程：

```
1. 识别冲突文件：git status 显示 "both modified" / "Unmerged paths"
2. 打开冲突文件，定位 <<<<<<< HEAD / ======= / >>>>>>> 三段
3. 人工裁决：保留哪段 / 合并哪段 / 删除哪段，并删掉冲突标记
4. git add <文件>  标记为已解决
5a. merge 场景：git commit 完成合并
5b. rebase 场景：git rebase --continue；想放弃本次变基：git rebase --abort
```

**伪代码（rebase 冲突期间的处理循环）**：

```
while 有冲突文件:
    编辑解决 -> git add
git rebase --continue   # 或 --skip 跳过该提交，--abort 整体放弃
```

> ⚠️ **易错点**：rebase 冲突时**不要用 `git merge --continue`**；merge 冲突时也不要 `git rebase --continue`，两者命令不同。rebase 会重写提交，**不要在共享分支上 rebase**。

### 1.4 rebase vs merge 对比

| 维度 | merge | rebase |
| --- | --- | --- |
| 历史形态 | 保留分叉，形成合并提交 | **线性历史**，提交被重放到新基点 |
| 是否重写提交 | 否 | **是（换 hash）** |
| 冲突次数 | 一次合并解决一次 | **每个被重放的提交都可能冲突** |
| 可追踪性 | 保留「何时合并了哪个分支」 | 丢失原分叉信息 |
| 共享分支安全 | ✅ 安全 | ❌ 危险（会改写他人已有提交） |
| 典型场景 | 合并 feature 回主干 | 拉取远端时 `pull --rebase` 保持线性 |

**黄金法则**：**绝不 rebase 已经 push 到共享仓库的分支**。个人未推送的分支可自由 rebase。

### 1.5 reset vs revert

| 命令 | 效果 | 历史 | 适用 |
| --- | --- | --- | --- |
| `git reset --hard <sha>` | HEAD 回退，工作区丢弃 | **重写历史**，reflog 可救 | 本地未推送提交 |
| `git reset --soft <sha>` | HEAD 回退，改动留在暂存区 | 重写历史 | 合并多个提交 |
| `git revert <sha>` | 生成一个「反向提交」 | **追加新提交，历史不变** | 已推送 / 线上回滚 |

**口诀：reset 回退「删掉过去」，revert 回退「否定过去」**。已 push 的提交回退必须用 revert，否则 push 会被拒绝（`non-fast-forward`）。

### 1.6 面试追问

**Q：rebase 和 merge 你生产环境怎么选？**
A：个人未推送分支用 rebase 保持整洁；合入主干用 merge（`--no-ff`）保留合并语义。共享分支只 merge 不 rebase。

**Q：代码已 push 又想撤销怎么办？**
A：`git revert <sha>` 提交一个反向提交再 push；不要 reset 后强推（`push -f`），会破坏协作者的本地历史。

**Q：cherry-pick 用在什么场景？**
A：把某个修复/功能**单独**搬到其他分支（如把 hotfix 挑到 develop），避免整条 merge；代价是重复提交，历史里会出现两份相同改动。

**Q：stash 后如何找回？**
A：`git stash list` 查看，`git stash pop` 恢复并删除记录；`git stash apply` 保留记录；误删可用 `git fsck --lost-found` 或 reflog 找回。

> 💡 **面试点评**：会背命令清单只是及格；加分点是**场景判断**——「已推送回退用 revert、共享分支不 rebase、stash 与冲突解决流程」这类实战判断。

---

## 2. Linux 常用命令与现场排查

### 2.1 命令速查表

| 命令 | 用途 | 典型参数 |
| --- | --- | --- |
| `ps` | 查看进程快照 | `ps aux` 全量；`ps -ef` 标准格式；`ps aux \| grep python` 过滤 |
| `top` | 实时进程/负载监控 | `top -Hp <pid>` 看线程；`top -d 1` 1 秒刷新；`P`/`M` 按 CPU/内存排序 |
| `free` | 内存使用 | `free -h` 人类可读；`free -m` 以 MB 显示 |
| `df` | 磁盘分区占用 | `df -h`；`df -i` 看 inode |
| `du` | 目录/文件占用 | `du -sh *` 汇总当前目录各项；`du -h --max-depth=1` |
| `grep` | 文本匹配 | `grep -n pattern file`；`-i` 忽略大小写；`-v` 反向；`-r` 递归；`-E` 扩展正则 |
| `awk` | 文本按列处理 | `awk '{print $1}'`；`awk '$4 > 90 {print}'` 条件过滤 |
| `sed` | 流编辑 | `sed -n '5,10p'` 打印行段；`sed -i 's/old/new/g'` 原地替换 |
| `netstat` | 网络连接/端口 | `netstat -tlnp` 监听端口+进程；`-u` UDP；`ss` 是其现代替代 |
| `curl` | 发送 HTTP 请求 | `curl -i` 带头；`-X POST`；`-d '{}'`；`-m 5` 超时；`-w "%{time_total}"` 计时 |
| `tail` | 查看文件尾部 | `tail -f app.log` 实时跟踪；`-n 200` 指定行数 |

### 2.2 五个「现场排查」场景

**场景① 端口被占用（8080 起不来服务）**

```bash
ss -tlnp | grep 8080          # 或 netstat -tlnp | grep 8080
lsof -i :8080                 # 直接列出占用 8080 的进程
kill -9 <PID>                 # 确认是僵尸进程后强杀；否则先 kill <PID>
```

**场景② 内存爆了（free 显示几乎耗尽 / 服务 OOM）**

```bash
free -h                        # 确认 available 是否告急
ps aux --sort=-%mem | head -5  # 找出内存占用 TOP 进程
cat /proc/<pid>/smaps | grep -E "Rss|Swap"   # 细看单进程内存分布
dmesg -T | grep -i "out of memory"           # 看内核 OOM-Killer 记录
# 定位到进程后：优化或重启，必要时调 ulimit / 换机器
```

**场景③ 磁盘满了（df 显示 100%，写入报 no space）**

```bash
df -h                           # 定位哪个分区满了
du -sh /* 2>/dev/null | sort -rh | head      # 找大目录（注意 / 通配要加 -x 避免跨分区）
find / -xdev -size +1G -type f 2>/dev/null   # 找大文件
lsof +L1 2>/dev/null             # 找已被删除但仍占空间的「幽灵文件」
# 清理后 df -h 确认；另查 df -i 排除 inode 耗尽
```

**场景④ 进程假死（top 正常但请求全部超时 / 线程卡死）**

```bash
ps -eo pid,ppid,stat,cmd | grep <服务名>      # 看状态：D(不可中断)/Z(僵尸)/T(停止)
top -Hp <pid>                                  # 看具体线程 CPU/状态
cat /proc/<pid>/task/<tid>/status | grep -E "State|voluntary_ctxt"
jstack <pid> | grep -A 20 "RUNNABLE"           # Java 场景看线程栈
# D 状态：多为磁盘/网络 IO 卡死；Z 状态：父进程未 wait，找父进程处理
```

**场景⑤ 日志过滤（从 GB 级日志找错误与统计）**

```bash
grep -n "ERROR\|Exception" app.log | tail -50         # 过滤错误
tail -f app.log | grep --line-buffered "timeout"      # 实时跟踪指定关键词
awk '$9 ~ /^5[0-9][0-9]$/ {print $9}' access.log | sort | uniq -c    # 统计 5xx 数量
sed -n '/2024-06-01 10:00/,/2024-06-01 10:05/p' app.log              # 按时间段切片
awk '{print $1}' access.log | sort | uniq -c | sort -rn | head -10   # Top10 访问 IP
```

### 2.3 面试追问

**Q：ps 和 top 的区别？**
A：ps 是**一次性快照**，top 是**实时刷新**（还能看负载、按 CPU/内存排序）；排查历史进程用 ps，监控实时状态用 top。

**Q：df 和 du 的区别？**
A：df 看**分区**整体使用量（从文件系统统计），du 从目录遍历**累加**文件大小；du 更准但慢，df 与 du 数字不一致常因「已删除但仍被进程占用」的文件。

**Q：awk 和 sed 各自擅长什么？**
A：awk 是**按列处理 + 编程**（字段、统计、条件），sed 是**按行流编辑**（打印区间、替换、删除行）。配合 grep（按行匹配）构成三件套。

**Q：排查 CPU 100% 的命令串？**
A：`top -H -p <pid>` 找烧 CPU 的线程 → `strace -p <tid>` 看系统调用 → 结合代码定位死循环/GC/自旋；Python 场景可用 `py-spy dump --pid <pid>` 直接拿 Python 栈。

> 💡 **面试点评**：命令要**成组记忆**（场景化输出命令串），面试官最反感「我只用过 ls/cd」。能说出 `ss` 替代 `netstat`、`lsof +L1` 找幽灵文件这类进阶用法是加分项。

---

## 3. pytest 测试框架

### 3.1 核心概念

**pytest** 是 Python 主流测试框架，特点：**函数式用例**（无需类）、自动发现（`test_*.py` / `test_*` 函数）、丰富断言（纯 `assert`，失败时自动显示上下文）、强大 fixture 体系。


![git_branch.png](images/git_branch.png)


*配图来源：本仓库 matplotlib 绘制（`images/git_branch.png`）*


In [ ]:
# test_demo.py —— 最简用例
def add(a, b):
    return a + b

def test_add():
    assert add(1, 2) == 3          # 直接 assert，失败自动展示 diff


运行：`pytest -v`（详细）/ `pytest test_demo.py::test_add`（单个）/ `pytest -x`（失败即停）/ `pytest --pdb`（失败进调试器）。

### 3.2 fixture 原理与 scope

**fixture** 是**可复用的测试依赖注入**：pytest 根据测试函数**参数名**自动查找同名 fixture，并**在测试前后执行 setup/teardown**（`yield` 之前是 setup，之后是 teardown）。


In [ ]:
import pytest

@pytest.fixture
def db_conn():
    print("\n[setup] 建立数据库连接")
    conn = {"connected": True}
    yield conn                        # 测试拿到的是 yield 出来的值
    print("[teardown] 关闭连接")      # 测试结束后自动执行

def test_query(db_conn):
    assert db_conn["connected"] is True


**scope 控制生命周期**：

| scope | 生命周期 | 适用 |
| --- | --- | --- |
| `function`（默认） | 每个测试函数独立 | 大部分场景 |
| `class` | 每个测试类一次 | 类内共享 |
| `module` | 每个模块一次 | 模块级共享资源 |
| `session` | 整个测试会话一次 | 数据库、昂贵连接 |


In [ ]:
@pytest.fixture(scope="session")
def big_cache():
    return {}                        # 整个测试会话只建一次


**fixture 依赖**：fixture 可以引用其他 fixture（参数名即依赖），pytest 自动拓扑排序执行；`autouse=True` 让所有测试自动使用（无参数依赖时）。


In [ ]:
@pytest.fixture
def user(db_conn):                  # 依赖 db_conn
    return {"id": 1, "name": "alice"}


### 3.3 参数化 parametrize

`@pytest.mark.parametrize` 把同一用例跑多组数据，避免复制代码：


In [ ]:
import pytest

@pytest.mark.parametrize("a,b,expected", [
    (1, 2, 3),
    (0, 0, 0),
    (-1, 1, 0),
    (100, 200, 300),
], ids=["case1", "zero", "neg", "big"])   # ids 自定义用例名
def test_add(a, b, expected):
    assert add(a, b) == expected


**笛卡尔积组合**：多个 parametrize 叠加 = 全组合；同一用例参数从 fixture 拿（`indirect=True`）也常见。

### 3.4 mock / monkeypatch

**monkeypatch** 是 pytest 内置 fixture，用于**运行时替换属性/环境变量**（如打桩网络请求）：


In [ ]:
# 打桩：不真实发 HTTP
def get_weather(city):
    import urllib.request
    with urllib.request.urlopen(f"http://api/weather/{city}") as r:
        return r.read().decode()

def test_get_weather(monkeypatch):
    def fake_urlopen(url):
        return io.StringIO('{"temp": 25}')
    monkeypatch.setattr("urllib.request.urlopen", fake_urlopen)   # 运行时替换
    assert "25" in get_weather("beijing")


**unittest.mock / pytest-mock**（`mocker` fixture）提供更强的 Mock 对象（断言调用次数/参数、自动清理）：


In [ ]:
# 需安装 pytest-mock：pip install pytest-mock
def test_calls(mocker):
    fake = mocker.patch("module.send_email")       # 自动 mock 并还原
    call_api()
    fake.assert_called_once_with(to="a@b.com")     # 断言调用参数


> ⚠️ **易错点**：patch 的路径要打**「被使用时」的模块**（`app.utils.send_email`）而非定义处；monkeypatch 改动**自动还原**，这是它比手动 setattr 安全的原因。

### 3.5 断言风格

- 原生 `assert`，失败自动 diff（pytest 会反编译语句展示左右值）；
- 异常断言：`pytest.raises(ZeroDivisionError)`；近似比较：`pytest.approx`；字符串包含：`in`。


In [ ]:
import pytest

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("b cannot be 0")
    return a / b

def test_divide_error():
    with pytest.raises(ZeroDivisionError, match="cannot"):
        divide(1, 0)

def test_approx():
    assert 0.1 + 0.2 == pytest.approx(0.3)      # 浮点近似比较


### 3.6 完整可运行示例


> 🔬 **示例代码（需在独立环境中运行）**：以下代码块依赖外部组件/独立进程，在 notebook 中直接执行会失败或挂起，请保存为 `.py` 文件后运行。

```python
# test_order.py —— 可直接 pytest 运行
import pytest
from unittest import mock

class Order:
    def __init__(self, items):
        self.items = items

    def total(self):
        return sum(i["price"] for i in self.items)

    def pay(self, gateway):
        return gateway.charge(self.total())

@pytest.fixture(scope="function")
def order():
    return Order([{"name": "apple", "price": 10},
                  {"name": "banana", "price": 5}])

@pytest.mark.parametrize("extra,expected", [([], 15), ({"name": "x", "price": 100}, 115)])
def test_total(order, extra, expected):
    order.items = order.items + extra
    assert order.total() == expected

def test_pay_mock(order, mocker):            # mocker 需要 pytest-mock
    fake_gw = mocker.Mock(charge=mocker.Mock(return_value="OK"))
    assert order.pay(fake_gw) == "OK"
    fake_gw.charge.assert_called_once_with(15)
```


### 3.7 面试追问

**Q：fixture 的 teardown 怎么保证一定执行？**
A：`yield` 之后的代码在测试结束**无论成败**都会执行（类似 finally）；也支持 `addfinalizer`。这是 pytest 比 `setup/teardown` 旧式写法更可靠的核心理由。

**Q：参数化用例失败如何定位哪组数据？**
A：pytest 会为每组生成带参数用例名（`ids` 可自定义）；`pytest -k` 过滤、`pytest -vv` 显示参数化全名。

**Q：mock 与真实依赖怎么权衡？**
A：单元测试尽量 mock 外部边界（网络/DB/时间）；集成测试走真实依赖。过度 mock 会让测试「测试自己的 mock」，失去意义。

> 💡 **面试点评**：能现场说清「fixture 参数名即依赖注入 + yield 做 teardown + scope 生命周期 + parametrize 组合」四件事，pytest 题基本无死角。

---

## 4. 常用设计模式与 Python 实现

### 4.1 单例模式（3 种写法）

**定义**：保证一个类**只有一个实例**，并全局可访问。适用：配置中心、连接池、日志器。

**写法一：`__new__` 拦截（最标准）**


In [ ]:
class Singleton:
    _instance = None
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

s1, s2 = Singleton(), Singleton()
print(s1 is s2)              # True


**写法二：装饰器**


In [ ]:
def singleton(cls):
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Config:
    pass

print(Config() is Config())   # True


**写法三：模块级单例（Python 惯用法）**


In [ ]:
# singleton_module.py
class _Logger:
    def __init__(self):
        self.level = "INFO"

logger = _Logger()             # 模块导入即单例，最 Pythonic


> ⚠️ **易错点**：单例**不是线程安全**的（`__new__` 竞态），多线程下需加锁；测试中单例难以隔离，常用工厂注入替代。面试常追问「单例的坏处」——答：全局可变状态、隐藏依赖、难以测试。

### 4.2 工厂模式（Factory）

**定义**：把「创建对象」的逻辑封装，客户端不依赖具体类。适用：产品族多、创建逻辑复杂（如根据配置创建不同模型/存储后端）。


In [ ]:
class Storage:
    def save(self, data): ...

class S3Storage(Storage):
    def save(self, data): return f"S3 saved {data}"

class LocalStorage(Storage):
    def save(self, data): return f"local saved {data}"

class StorageFactory:                     # 简单工厂
    @staticmethod
    def create(kind):
        if kind == "s3":
            return S3Storage()
        if kind == "local":
            return LocalStorage()
        raise ValueError(kind)

print(StorageFactory.create("s3").save("x"))     # S3 saved x


**抽象工厂 / 工厂方法**：把创建逻辑下沉到子类（`create` 是抽象方法），扩展新产品只需加子类，**符合开闭原则**。

### 4.3 观察者模式（Observer）

**定义**：**发布-订阅**——被观察者状态变化时自动通知所有订阅者。适用：事件系统、UI 回调、指标上报。


In [ ]:
class Subject:
    def __init__(self):
        self._observers = []

    def attach(self, obs): self._observers.append(obs)
    def detach(self, obs): self._observers.remove(obs)

    def notify(self, event):
        for obs in self._observers:
            obs.update(event)

class EmailNotifier:                         # 观察者 1
    def update(self, event): print(f"[email] {event}")
class LogNotifier:                           # 观察者 2
    def update(self, event): print(f"[log] {event}")

s = Subject()
s.attach(EmailNotifier())
s.attach(LogNotifier())
s.notify("user registered")
# [email] user registered
# [log] user registered


Python 内置替代：`asyncio` 事件、`Observable`（Rx）或直接用回调列表——观察者模式的本质就是**回调注册表**。

### 4.4 策略模式（Strategy）

**定义**：把可替换的算法族封装成对象，运行时切换。适用：多种排序/计价/压缩/路由算法。


In [ ]:
class QuickSort:
    def sort(self, data): return sorted(data)          # 简化示例
class MergeSort:
    def sort(self, data): return sorted(data, reverse=False)

class Sorter:                                          # 上下文：持有策略
    def __init__(self, strategy):
        self.strategy = strategy
    def run(self, data):
        return self.strategy.sort(data)

sorter = Sorter(QuickSort())
print(sorter.run([3, 1, 2]))                           # [1, 2, 3]
sorter.strategy = MergeSort()                          # 运行时切换策略


**与函数式对比**：Python 中策略常简化为**传函数**（`sorted(data, key=...)`）——策略模式本质是「依赖注入算法」；面试可指出「一等函数让策略模式退化为高阶函数」。

### 4.5 装饰器模式（结构型，区别于 Python 装饰器语法）

**定义**：**动态给对象添加职责**，用组合替代继承。适用：给类/对象叠加日志、鉴权、缓存而**不改动原类**。


In [ ]:
class TextComponent:
    def render(self): return "hello"

class BoldDecorator:                    # 装饰者：包装并增强
    def __init__(self, wrapped):
        self._wrapped = wrapped
    def render(self):
        return f"**{self._wrapped.render()}**"

t = BoldDecorator(TextComponent())
print(t.render())                       # **hello**
# 可无限叠加：BoldDecorator(ItalicDecorator(TextComponent()))


**与 Python 装饰器语法的关系**：语法糖装饰器本质也是「函数包装函数」，二者理念相同——**包装 + 透传 + 增强**。

### 4.6 设计模式对比表

| 模式 | 类型 | 核心思想 | Python 场景 |
| --- | --- | --- | --- |
| 单例 | 创建型 | 全局唯一实例 | 配置/连接池 |
| 工厂 | 创建型 | 创建逻辑封装 | 存储后端、模型工厂 |
| 观察者 | 行为型 | 发布-订阅解耦 | 事件、回调 |
| 策略 | 行为型 | 算法族可替换 | 排序/计价策略 |
| 装饰器 | 结构型 | 组合叠加职责 | 日志/鉴权/缓存包装 |

### 4.7 面试追问

**Q：单例和全局变量的区别？**
A：单例保证**唯一实例 + 统一入口**，可延迟创建、可继承控制；但仍属于**全局可变状态**，是两者共同的缺点。现代框架更推崇「依赖注入 + 容器管理生命周期」。

**Q：观察者模式有什么坑？**
A：订阅者**忘记反注册**会内存泄漏（被 Subject 强引用）；通知顺序不应被依赖；异常会中断通知链。Python 中注意**循环引用**导致 GC 回收问题。

**Q：策略模式在 Python 里怎么简化？**
A：算法少时直接传函数/lambda（一等公民），只有策略多、需配置与校验时才用类封装——「过度设计」也是面试要主动避免的点。

> 💡 **面试点评**：设计模式题的关键不是背 UML，而是**场景 + 权衡**。每个模式能说出「解决什么问题、Python 里的天然替代、有什么坑」即高分。

---

## 5. 性能分析与线上排查

### 5.1 分析工具四件套

**① timeit（微观基准）**：对代码片段精确计时（默认百万次），排除系统噪音：


In [ ]:
import timeit
print(timeit.timeit("[x for x in range(1000)]", number=10000))
print(timeit.timeit("list(range(1000))", number=10000))     # 后者明显更快


**② cProfile（函数级 CPU 剖析）**：记录每个函数调用次数与耗时，定位热点：

```bash
python -m cProfile -s cumtime my_script.py      # 按累计耗时排序
```


In [ ]:
import cProfile, pstats

cProfile.run("sum(range(1_000_000))", "prof.out")
pstats.Stats("prof.out").sort_stats("cumulative").print_stats(10)


**③ line_profiler（逐行剖析）**：`@profile` 装饰器标注函数，输出每行耗时（需 `pip install line_profiler`）：


In [ ]:
# 行级性能分析：@profile 装饰器来自 line_profiler（kernprof 运行）
# 这里提供降级定义，使 notebook 内可直接运行演示
try:
    from line_profiler import profile
except ImportError:
    def profile(fn):
        return fn          # 未安装 line_profiler 时退化为普通函数

@profile                     # kernprof -l -v my_script.py
def process(data):
    total = 0
    for x in data:           # 这行如果占大头，说明循环是瓶颈
        total += x * x
    return total

print("process([1,2,3,4]) =", process([1, 2, 3, 4]))


**④ memory_profiler（内存剖析）**：`@profile` 看每行内存增量（需 `pip install memory_profiler`）；`tracemalloc` 用于精确定位**内存泄漏**快照对比：


In [ ]:
import tracemalloc
tracemalloc.start()
a = [list(range(1000)) for _ in range(100)]     # 模拟分配
snap1 = tracemalloc.take_snapshot()
b = [list(range(1000)) for _ in range(100)]
snap2 = tracemalloc.take_snapshot()
top = snap2.compare_to(snap1, 'lineno')
for stat in top[:3]:
    print(stat)


### 5.2 排查流程

**CPU 100% 排查步骤**：

```
1. top → 找高 CPU 进程 PID
2. top -Hp <pid> → 找具体线程 TID
3. 采样线程栈：strace -p <tid>（系统调用） / py-spy dump --pid <pid>（Python 栈）
4. 结合代码定位：死循环 / 忙等 / 过度 GC / 低效算法
5. 修复后对比验证（cProfile 前后 profile 对比）
```

**OOM / 内存暴涨排查步骤**：

```
1. free -h + dmesg -T | grep -i oom → 确认 OOM-Killer
2. ps aux --sort=-%mem → 找出大内存进程
3. tracemalloc 快照对比 → 判断「泄漏」还是「正常增长」
4. 排查方向：缓存无上限 / 全局 list 累积 / 循环引用未回收 / 大对象常驻
5. 修复：限流缓存、分页处理、gc.collect 兜底、减小批大小
```

**慢接口排查步骤**：

```
1. 确认现象：curl -w "%{time_connect} %{time_starttransfer}" 分段计时
2. 区分阶段：网络耗时？服务处理耗时？下游耗时？
3. 服务端：cProfile 定位函数热点；慢 SQL 用 EXPLAIN 分析索引
4. 并发视角：连接池耗尽 / 锁竞争 / 单线程事件循环被阻塞
5. 验证：压测（wrk/ab）对比修复前后 p99 延迟
```

### 5.3 对比表

| 工具 | 粒度 | 成本 | 定位的问题 |
| --- | --- | --- | --- |
| `timeit` | 代码片段 | 极低 | 某段代码谁快 |
| `cProfile` | 函数级 | 中（有开销） | 热点函数、调用次数 |
| `line_profiler` | 行级 | 高 | 具体哪行是瓶颈 |
| `memory_profiler` | 行级内存 | 高 | 内存分配热点 |
| `tracemalloc` | 分配点 | 中 | 内存泄漏 |

### 5.4 面试追问

**Q：cProfile 的 overhead 会影响结果吗？**
A：会。它给每个函数插桩，总耗时被放大（尤其高频小函数）；结论应看**相对占比**而非绝对时间。低开销替代：`py-spy`（采样式，不插桩）。

**Q：线上没装 profile 工具怎么排查？**
A：用 `py-spy`（采样 Python 栈，无需改代码）；或临时加日志打点（埋点计时）；或导出 profile 到文件离线分析。**绝不在线上直接跑完整 cProfile**。

**Q：内存泄漏 vs 内存增长怎么区分？**
A：泄漏 = 回收不了（tracemalloc 快照间持续增长且无法解释）；增长 = 正常业务（缓存、池、批处理）。判断标准：**跑两轮相同负载，比较内存快照差异**。

> 💡 **面试点评**：把「工具链（timeit→cProfile→line_profiler→memory_profiler）」与「排查流程（top→定位→采样→验证）」串成方法论，面试官会追问「具体一次你怎么做的」，提前准备一个真实案例。

---

## 6. 进程管理工具与 Docker

### 6.1 supervisor / systemd

**supervisor**：Python 生态的**进程管理守护进程**，负责**拉起、监控、自动重启**业务进程（非 init 系统，跑在用户态）：

```ini
; /etc/supervisor/conf.d/app.conf
[program:app]
command=python -m app.server          ; 启动命令
directory=/srv/app
autostart=true                        ; 开机自动启动
autorestart=true                      ; 崩溃自动重启
stderr_logfile=/var/log/app.err       ; 日志
stdout_logfile=/var/log/app.out
numprocs=1
```

```bash
supervisorctl status        # 查看全部
supervisorctl restart app   # 重启指定
supervisorctl tail -f app   # 跟踪日志
```

**systemd**：现代 Linux 的**系统级 init 与服务管理器**，管理系统服务、开机自启、依赖、资源限制：

```ini
# /etc/systemd/system/app.service
[Unit]
Description=My App
After=network.target

[Service]
ExecStart=/usr/bin/python /srv/app/server.py
Restart=always                 # 崩溃自动重启
RestartSec=3
User=www
LimitNOFILE=65535              # 文件描述符上限

[Install]
WantedBy=multi-user.target
```

```bash
systemctl daemon-reload        # 修改配置后重载
systemctl start/stop/restart app
systemctl status app           # 查看状态与最近日志
systemctl enable app           # 开机自启
journalctl -u app -f           # 跟踪服务日志
```

**对比表**：

| 维度 | supervisor | systemd |
| --- | --- | --- |
| 定位 | 应用级进程管家（用户态） | 系统级服务管理（PID 1） |
| 自启 | 依赖外部 init 拉它 | ✅ 原生开机自启 |
| 资源限制 | 弱 | 强（Limit*、cgroup） |
| 日志 | 文件 | journald 统一收集 |
| 适用 | Python/脚本进程、无 systemd 环境 | 现代 Linux 全场景 |

### 6.2 Docker 基础命令

**Docker** 用**容器**（共享内核的隔离进程 + 镜像层文件系统）打包应用。核心命令：

| 命令 | 作用 | 典型用法 |
| --- | --- | --- |
| `docker build` | 按 Dockerfile 构建镜像 | `docker build -t myapp:1.0 .` |
| `docker run` | 启动容器 | `docker run -d -p 8080:80 --name web myapp:1.0` |
| `docker ps` | 查看容器 | `docker ps -a` 含已停止；`docker ps -f name=web` 过滤 |
| `docker logs` | 查看容器日志 | `docker logs -f --tail 100 web` |
| `docker exec` | 进入运行中容器 | `docker exec -it web bash` |
| `docker stop/rm` | 停止/删除容器 | `docker rm -f web` |
| `docker images` / `rmi` | 镜像管理 | `docker rmi myapp:old` |
| `docker compose up` | 多容器编排 | `docker compose up -d` |

**Dockerfile 最小示例**：

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt     # 依赖层：改动少时利用层缓存
COPY . .
CMD ["python", "app.py"]
```

```bash
docker build -t myapp:1.0 .
docker run -d -p 8000:8000 --restart=unless-stopped myapp:1.0
docker ps
docker logs -f $(docker ps -q -f ancestor=myapp:1.0)
docker exec -it $(docker ps -q -f ancestor=myapp:1.0) bash
```

> ⚠️ **易错点**：`docker exec` 进的是**运行中容器**，`docker run` 是**新容器**；`docker logs` 只看 **stdout/stderr**，进程直接写文件的日志看不到；容器内进程 PID 1 退出 = 容器退出（要前台运行应用）。

### 6.3 面试追问

**Q：容器和虚拟机的区别？**
A：虚拟机有**独立内核**（Hypervisor 虚拟化硬件），隔离强、开销大；容器**共享宿主机内核**（cgroup + namespace 隔离），秒级启动、镜像分层复用。隔离强度弱于虚拟机（逃逸风险更高）。

**Q：supervisor 的 autorestart 与 systemd 的 Restart 区别？**
A：本质相同——守护进程监控子进程退出码自动拉起；systemd 还能做**资源限制、开机自启、日志统一**。Docker 里则用 `--restart` 策略（`unless-stopped`/`always`）。

**Q：线上容器如何查问题？**
A：`docker logs` 看输出 → `docker exec` 进容器用 `ps/top` 看资源 → `docker inspect` 看配置/挂载 → 宿主机 `docker stats` 看资源占用。注意容器内可能没有调试工具（需挂载或换镜像）。

> 💡 **面试点评**：Docker 题常考「镜像与容器区别（类 vs 实例）、`-p` 端口映射、exec 进容器排障、重启策略」四件事；能把「容器共享内核」讲透即通过。

---

## 附录：速记卡（一句话 × 6 主题）

1. **Git**：merge 保留分叉、rebase 线性重写；共享分支禁 rebase；已推送回退用 revert；冲突 = 三段标记 + add + continue。
2. **Linux**：`ss/ps/top/free/df/du/grep/awk/sed/curl/tail` 成组记忆，排查按「定位进程 → 采样 → 验证」走流程。
3. **pytest**：fixture 参数名注入 + yield 做 teardown + scope 控生命周期；parametrize 多组数据；monkeypatch/mocker 打桩。
4. **设计模式**：单例 3 写法、工厂解耦创建、观察者发布订阅、策略算法可换、装饰器组合加职责——**每个都要能说场景与坑**。
5. **性能分析**：timeit→cProfile→line_profiler→memory_profiler 递进；线上用 py-spy 采样；排查先 top 定位再取样。
6. **进程管理**：supervisor 应用级管家，systemd 系统级，Docker 容器共享内核、镜像分层、`exec` 进容器排障。



## 附录二：追加内容——工程实战补充（秋招冲刺）

> 本附录为原笔记的追加章节（B1~B9），覆盖 Git 场景命令、pytest fixture 进阶、Linux 排查指标、
> awk/sed 文本处理、设计模式选型、性能基准、Docker、代码审查。每节带「面试追问 Q&A」与
> 「易错点/面试点评」。

### B1 Git 后悔药：checkout / reset --hard / revert / stash

开发时最怕「改错了」。四种回归命令的**适用场景**完全不同，面试官常让你对比：

| 命令 | 作用对象 | 效果 | 历史是否保留 | 典型场景 |
| --- | --- | --- | --- | --- |
| `git checkout -- <file>` | 工作区单文件 | 丢弃未暂存的本地修改 | 是 | 改乱了还没 add |
| `git restore <file>` | 工作区 | 同左（Git 2.23+ 推荐写法） | 是 | 语义化替代 checkout |
| `git reset --hard <commit>` | 分支指针+暂存区+工作区 | **移动**分支指针，丢弃其后的提交 | **否**（有风险） | 本地提交写错，还没 push |
| `git revert <commit>` | 生成一个新的反向提交 | 前向「撤销」，不移动指针 | 是（新增提交） | 已 push 的提交，安全回滚 |
| `git stash` | 工作区改动 | 暂存起来清空工作区，可恢复 | 是 | 切换分支前先藏起来 |

口诀：**没 push 用 reset，已 push 用 revert；只动工作区用 checkout/restore；临时打断用 stash。**

**面试追问 Q&A**

**Q：`git reset --hard` 后还能找回提交吗？**
A：能——提交对象还在的话，`git reflog` 能找到 reset 前的 HEAD，`git reset --hard <原hash>`
就能救回来。但 reflog 有保留期（默认约 90 天，之后被 gc 清理），换机器/克隆后也没有。
所以：**push 过的历史永远别 reset，用 revert**。

**Q：`git revert` 遇到冲突怎么处理？**
A：revert 本质是应用一个「反向提交」，冲突时 `git status` 看冲突文件，手动解决后
`git add` + `git revert --continue` 完成。revert 不删原提交，不会重写团队共享历史。

> ⚠️ **易错点**：`git checkout` 既能切分支又能丢弃文件修改——写全提交号/分支名是切分支，
> `git checkout -- <file>` 才是「丢弃修改」；漏了 `--` 或写反顺序会误切分支。另外
> `reset --hard` 会**同时**清掉暂存区与工作区——只想动暂存区用 `git reset`（不带 --hard）。


In [ ]:
# B1 实战：在临时仓库里演示 checkout / reset --hard / revert / stash
# （不碰真实仓库：全部在临时目录中操作，结束后自动清理）
import os
import shutil
import subprocess
import tempfile

GIT = "git"
ENV = dict(os.environ)
ENV["LC_ALL"] = "C"                        # 强制英文输出，便于解析
ENV["LANGUAGE"] = "C"


def git(*args, cwd):
    r = subprocess.run([GIT, *args], cwd=cwd, env=ENV, capture_output=True)
    out = r.stdout.decode("utf-8", errors="replace").strip()
    err = r.stderr.decode("utf-8", errors="replace").strip()
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} 失败: {err or out}")
    return out


if shutil.which(GIT) is None:
    print("未检测到 git，跳过演示（其余内容不受影响）")
else:
    repo = tempfile.mkdtemp(prefix="git_demo_")
    try:
        # ① 初始化并提交 v1
        git("init", "-q", cwd=repo)
        git("config", "user.email", "demo@example.com", cwd=repo)
        git("config", "user.name", "demo", cwd=repo)
        git("config", "core.autocrlf", "false", cwd=repo)
        with open(os.path.join(repo, "app.py"), "w", encoding="utf-8") as f:
            f.write('print("v1")\n')
        git("add", ".", cwd=repo)
        git("commit", "-m", "v1: 初版", cwd=repo)
        print("1) 初始提交:", git("log", "--oneline", cwd=repo).replace("\n", " | "))

        # ② restore：丢弃工作区未暂存修改
        with open(os.path.join(repo, "app.py"), "w", encoding="utf-8") as f:
            f.write('print("改坏了")\n')
        print("2) 改坏后 diff:", git("diff", "--stat", cwd=repo).replace("\n", " | "))
        git("restore", "app.py", cwd=repo)
        status = git("status", "--short", cwd=repo)
        print("   restore 后状态:", repr(status) if status else "干净")

        # ③ reset --hard：丢弃本地提交（未 push）
        with open(os.path.join(repo, "app.py"), "w", encoding="utf-8") as f:
            f.write('print("v2 bug")\n')
        git("commit", "-am", "v2: 有问题的提交", cwd=repo)
        print("3) reset 前:", git("log", "--oneline", cwd=repo).replace("\n", " | "))
        git("reset", "--hard", "HEAD~1", cwd=repo)
        print("   reset --hard HEAD~1 后:",
              git("log", "--oneline", cwd=repo).replace("\n", " | "))

        # ④ revert：已 push 的场景，前向生成反向提交
        with open(os.path.join(repo, "app.py"), "w", encoding="utf-8") as f:
            f.write('print("v3 new feature")\n')
        git("commit", "-am", "v3: 新功能", cwd=repo)
        print("4) revert 前 tip:", git("log", "--oneline", "-1", cwd=repo))
        git("revert", "--no-edit", "HEAD", cwd=repo)
        print("   revert 后 tip:", git("log", "--oneline", "-1", cwd=repo))
        print("   历史（revert 追加而非改写）:")
        for line in git("log", "--oneline", cwd=repo).splitlines():
            print("     ", line)

        # ⑤ stash：临时藏起改动，再恢复
        with open(os.path.join(repo, "app.py"), "w", encoding="utf-8") as f:
            f.write('print("未完成的改造")\n')
        git("stash", "push", "-m", "临时藏起", cwd=repo)
        status = git("status", "--short", cwd=repo)
        print("5) stash 后 status:", repr(status) if status else "干净")
        print("   stash list:", git("stash", "list", cwd=repo))
        git("stash", "pop", cwd=repo)
        print("   stash pop 后 diff:",
              git("diff", "--stat", cwd=repo).replace("\n", " | "))
    finally:
        shutil.rmtree(repo, ignore_errors=True)
    print("临时仓库已清理")


### B2 Git 分支工作流实战：branch → PR → merge

小团队/开源协作的标准流程（功能分支模型，面试可整段口述）：

```
1. git checkout -b feat/xxx origin/main      # 从主线拉新分支
2. 开发 + 本地提交若干
   git add -A && git commit -m "feat: xxx"
3. git push -u origin feat/xxx               # 推送远端并建立跟踪
4. 在 GitHub/GitLab 发起 Pull Request(MR)，指定 reviewers
5. Review：CI 跑测试 + 人审代码，按意见修改后追加提交
6. Merge：通过后合入 main（squash / merge commit / rebase 三选一）
7. git checkout main && git pull && git branch -d feat/xxx   # 清理本地分支
```

**三种合入方式对比**：

| 方式 | 历史形态 | 适用 |
| --- | --- | --- |
| `git merge feat` | 保留分叉 + 一个合并提交 | 想保留「功能分支」痕迹 |
| `git merge --squash feat` | 压缩成一个提交 | 功能颗粒度小、历史要线性 |
| `rebase 后再 git merge --ff-only` | 完全线性 | 团队规范要求线性历史 |

**面试追问 Q&A**

**Q：PR review 后要改，用新 commit 还是 amend？**
A：未合并前都可以，但规范是**追加 commit**（保留 review 过程，评审人和 CI 增量对比）；
`amend`/`rebase -i` 改写历史，适合**合并前最后整理**。已 push 到共享分支的提交**不要 amend**，
会让别的协作者 pull 时冲突。

**Q：feature 分支落后 main 怎么办？**
A：两种：`git merge main`（合入主线，保留分叉）或 `git rebase main`（把分支重放到主线顶部，
历史线性）。合入前先 rebase 再 `merge --ff-only` 即官方推荐的「rebase 后快进」工作流。
铁律：**别人已经拉取过的分支不要 rebase**。

> ⚠️ **易错点**：① 忘了 `git fetch` 直接 merge 的是**本地过期的 main**——先 fetch/pull；
> ② `git checkout -b` ≡ `git branch` + `git checkout`，讲流程时顺带说出等价关系加分；
> ③ push 前先 `git pull --rebase`，避免产生无意义的 merge commit；④ CI 必须在 merge 前
> 全绿——merge 了红 CI 的 commit 等于把锅甩给全队。



### B3 pytest fixture 进阶：autouse / conftest.py / 依赖链

原笔记 3.2 讲了 fixture 基础与 scope，面试更爱追问这三个进阶机制：

**① fixture 依赖 fixture**：fixture 的参数名写成另一个 fixture 名，pytest 自动按拓扑序
构造并注入，**销毁顺序与创建相反**（后创建先销毁）。

**② `autouse=True`**：同作用域内的测试**自动使用**该 fixture，无需在参数里声明。
典型用途：全局计时、日志配置、环境变量打桩、数据库清理。

**③ `conftest.py`**：共享 fixture 的存放文件，**不需要 import**——pytest 自动发现。
conftest 按目录层级作用：放在 `tests/` 根目录，`tests/` 下所有测试可见；子目录的 conftest
只对子树生效。共享 fixture 的**第一选择**就是 conftest.py。

**面试追问 Q&A**

**Q：fixture 里的 `request` 参数是什么？**
A：内置 fixture，携带「正在使用它的测试」的上下文：`request.param`（indirect 参数化）、
`request.node.name`（测试名）、`request.config`（命令行配置）。工厂型 fixture 常用它
区分不同调用方。

**Q：测试想「改 fixture 的返回值」怎么办？**
A：做不到直接改——fixture 是注入而非全局变量。需要「每个参数一个结果」用 `indirect=True`
参数化；需要「每次调用可配置」就把 fixture 做成**工厂**（yield 一个函数）。

> ⚠️ **易错点**：① teardown 必须写在 **yield 之后**；② autouse fixture 与普通 fixture
> 的执行顺序别靠记忆——需要精确顺序时**显式声明依赖**最可靠；③ conftest.py 被改动会让
> pytest 重新收集整个会话，别在里面放重逻辑（联网初始化之类）。


In [ ]:
# B3 实战：把测试写进临时文件，用 pytest 子进程运行验证
# 演示：fixture 依赖链、autouse 计时、request 上下文、工厂型 fixture
import os
import shutil
import subprocess
import sys
import tempfile
import textwrap

test_src = textwrap.dedent("""
    import pytest
    import time

    # ---- 常量配置（session 级：整个会话只创建一次）----
    @pytest.fixture(scope="session")
    def config():
        return {"timeout": 3, "retries": 2}

    # ---- fixture 依赖 fixture：client 依赖 config ----
    @pytest.fixture
    def client(config):
        print(f"[setup] 创建带 config 的客户端: {config}")
        yield {"base_url": "http://192.0.2.1/api"}
        print("[teardown] 关闭客户端")

    # ---- autouse：每个测试自动计时，无需显式声明 ----
    @pytest.fixture(autouse=True)
    def timer(request):
        t0 = time.perf_counter()
        yield
        print(f"  [timer] {request.node.name} 耗时 {time.perf_counter() - t0:.4f}s")

    # ---- 工厂型 fixture：yield 一个函数而非对象 ----
    @pytest.fixture
    def make_user():
        created = []

        def _make(uid, name):
            u = {"uid": uid, "name": name}
            created.append(u)
            return u

        yield _make
        print(f"[teardown] 清理 {len(created)} 个创建的用户")

    def test_ping(client):
        assert client["base_url"].endswith("/api")

    def test_user_flow(client, make_user):
        u1 = make_user(1, "alice")
        u2 = make_user(2, "bob")
        assert u1["name"] + u2["name"] == "alicebob"
""")

tmp = tempfile.mkdtemp(prefix="pytest_adv_")
try:
    test_file = os.path.join(tmp, "test_adv.py")
    with open(test_file, "w", encoding="utf-8") as f:
        f.write(test_src)

    env = dict(os.environ)
    env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"     # 提速：跳过第三方插件
    r = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", test_file],
        capture_output=True, text=True, env=env,
    )
    print(r.stdout)
    if r.stderr:
        print("[stderr]", r.stderr.strip())
    print(f"pytest 退出码: {r.returncode}（0 表示全部通过）")
finally:
    shutil.rmtree(tmp, ignore_errors=True)


### B4 Linux 排查实战：读懂 top / free -h

**top 首部三行解读**（面试必背）：

```
top - 10:15:01 up 3 days,  2:14,  1 user,  load average: 0.55, 0.31, 0.20
Tasks: 231 total,   1 running, 230 sleeping,   0 stopped,   0 zombie
%Cpu(s):  4.2 us,  1.3 sy,  0.0 ni, 93.7 id,  0.5 wa,  0.2 hi,  0.1 si,  0.0 st
MiB Mem :  31902.4 total,  21442.1 free,   7621.5 used,   4838.8 buff/cache
```

- **load average**：最近 1/5/15 分钟**就绪队列平均长度**（含等待 CPU 的线程）。
  判断：多核机器看 **load / 核数**（>1 说明有排队）；1 分钟值易抖动，15 分钟值看趋势。
- **us / sy**：用户态 / 内核态 CPU 占比；**wa** 是 IO 等待——wa 高说明磁盘/网络拖后腿；
  **id** 空闲；st 是虚拟机被宿主机抢占。
- **%MEM**：进程 RSS / 物理内存总量；RSS 含共享库，多进程会重复计数。
- **free -h**：`used` 包含 buff/cache；**`available` 才是「新程序还能用多少」**。

**面试追问 Q&A**

**Q：load average 为 3 的机器一定过载吗？**
A：不一定，要看核数：8 核机器 load=3 还很闲；单核机器 load=3 就是 3 倍过载。严谨说法是
**「每核 load」= load / 核数 > 1 说明有进程排队**。判趋势看 15 分钟值，1 分钟值易受瞬时任务影响。

**Q：free -h 里 used 快满了，但 available 还有不少，怎么回事？**
A：`used`（`total - free - buff/cache`）把 **buff/cache（页缓存、可回收的文件缓存）也算进去**；
`available` 估算「回收缓存后新程序还能拿到多少」。Linux 把空闲内存拿去做文件缓存是**特性**，
只要 available 充足就没有内存危机——**判断依据看 available，别只看 used**。

> ⚠️ **易错点**：① 把 load average 当 CPU 使用率是高频错误（它是**队列长度**不是百分比）；
> ② 把 used 当「真占用」忽略 buff/cache；③ top 的 %CPU 是累计平均而非实时——排查 CPU 飙升
> 先 `top -Hp <pid>` 看线程；④ 用 `ps aux --sort=-%cpu | head` 快速定位大户进程。


In [ ]:
# B4 实战：用 psutil 复刻 top / free 的关键指标（跨平台，语义与 Linux 一致）
import os
import time

import psutil

print("【load average】(Windows 上 psutil 返回 (0, 0, 0)；Linux/Mac 为真实值)")
try:
    print(f"  load average 1/5/15 min: {psutil.getloadavg()}")
except NotImplementedError:
    print("  当前系统不支持 load average")
print(f"  CPU 逻辑核数: {psutil.cpu_count()}，物理核数: {psutil.cpu_count(logical=False)}")

print()
print("【模拟 top 的 %Cpu(s)】先预热一次再采样")
psutil.cpu_percent(interval=None)                     # 第一次调用返回 0，用于预热
time.sleep(0.3)
per_core = psutil.cpu_percent(interval=1.0, percpu=True)
total = sum(per_core) / len(per_core)
print(f"  总使用率: {total:.1f}%")
print(f"  每核使用率: {['%.0f%%' % c for c in per_core]}")

print()
print("【free -h 的 Python 视角】(单位 GiB；used 含页缓存，看 available)")
mem = psutil.virtual_memory()
swap = psutil.swap_memory()
GB = 1024 ** 3
print(f"  Mem total : {mem.total / GB:8.2f} GiB")
print(f"  Mem used  : {mem.used / GB:8.2f} GiB   (含 buff/cache)")
print(f"  Mem avail : {mem.available / GB:8.2f} GiB   <- 判断紧张看它")
print(f"  使用率     : {mem.percent:.1f}%")
print(f"  Swap total/used: {swap.total / GB:.2f} / {swap.used / GB:.2f} GiB")

print()
print("【top 进程视图：按 CPU% 排序取前 5】")
procs = []
for p in psutil.process_iter(["pid", "name", "memory_percent"]):
    try:
        p.cpu_percent(None)                           # 预热每个进程
        procs.append(p)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        continue
time.sleep(1.0)
rows = []
for p in procs:
    try:
        rows.append((p.info["pid"], p.cpu_percent(None),
                     p.memory_percent(), p.info["name"]))
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        continue
rows.sort(key=lambda r: r[1], reverse=True)
print(f"  {'PID':>6}  {'CPU%':>5}  {'MEM%':>5}  名称")
for pid, cpu, mempct, name in rows[:5]:
    print(f"  {pid:>6}  {cpu:>5.1f}  {mempct:>5.1f}  {name}")

print()
print("【当前 Python 进程自身内存】（对应 %MEM）")
me = psutil.Process(os.getpid())
print(f"  pid={me.pid} RSS={me.memory_info().rss / 1024 / 1024:.1f} MB, "
      f"VMS={me.memory_info().vms / 1024 / 1024:.1f} MB")

### B5 awk / sed 文本处理三例（日志提取、批量替换、统计）

awk 按**列**加工，sed 按**行**修改——分工不同。三个实战例子（Linux/Mac 可直接跑，
示例为简化 Nginx 格式）。**列号提醒**：方括号时间 `[10/May/2024:10:00:01 +0800]`
含空格，awk 默认按空白拆列时会拆成 `$4`/`$5` 两列，所以 **状态码在第 9 列（$9）、
路径在第 7 列（$7）**：

**例 1 提取（awk）：过滤 5xx 请求的 IP、时间、状态码**
```bash
awk '$9 >= 500 {print $1, $4, $9}' access.log
```

**例 2 批量替换（sed）：旧域名换新域名**
```bash
sed -i 's/old\.example\.com/new\.example\.com/g' config/*.conf
# -i 原地修改；g 全局替换；\. 转义点号避免匹配任意字符
```

**例 3 统计（awk + sort）：每个接口 5xx 次数降序**
```bash
awk '$9 >= 500 {cnt[$7]++} END {for (u in cnt) print cnt[u], u}' access.log |
  sort -rn | head -10
# awk 数组即关联数组；END 块在读完所有行后执行
```

**面试追问 Q&A**

**Q：sed 和 awk 的适用边界？**
A：**按行做全局性修改**（替换/删除/插入）→ sed；**按列做结构化处理**（取字段、条件过滤、
聚合统计）→ awk。awk 自带关联数组与循环，能出简单报表；更复杂就上 Python/perl。
「sed 处理行、awk 处理列、Python 处理复杂逻辑」是标准答案。

**Q：正则引擎对命令有影响吗？**
A：有。**sed/awk 默认用 POSIX 基本正则（BRE）**：`+`、`?`、`|`、`()` 需要转义成
`\+`、`\(`；`sed -E` / `awk --posix` 才是扩展正则（ERE）。`s/a+/b/` 在 BRE 里
匹配不到任何内容却**不报错**——替换不生效先查转义。

> ⚠️ **易错点**：① `sed -i` 在 macOS（BSD）要写 `-i ''`，Linux GNU 版直接 `-i`；
> ② 生产环境**先不带 -i 输出验证**再原地替换，或先 `sed -i.bak` 留备份；
> ③ awk 默认按**空白**拆列，字段含空格（如上例的日期）要意识到列号会偏移，
> 或改用 `-F'|'` 等显式分隔符；④ 别拿 sed 处理二进制/超大文件。Windows 无 awk/sed——
> 下面用 Python 等价实现同一套处理思想。


In [ ]:
# B5 实战：模拟 access.log，用 Python 实现「awk/sed 三例」的等价逻辑
import re
from collections import Counter

log = """\
192.168.1.10 - - [10/May/2024:10:00:01 +0800] "GET /api/order HTTP/1.1" 200 1024 "/" "UA-1"
192.168.1.11 - - [10/May/2024:10:00:02 +0800] "GET /api/user HTTP/1.1" 500 0 "/" "UA-2"
192.168.1.12 - - [10/May/2024:10:00:03 +0800] "POST /api/order HTTP/1.1" 503 0 "/" "UA-1"
192.168.1.10 - - [10/May/2024:10:00:04 +0800] "GET /api/user HTTP/1.1" 200 512 "/" "UA-1"
192.168.1.13 - - [10/May/2024:10:00:05 +0800] "GET /api/pay HTTP/1.1" 502 0 "/" "UA-3"
192.168.1.14 - - [10/May/2024:10:00:06 +0800] "GET /api/order HTTP/1.1" 404 128 "/" "UA-2"
"""
LINES = [line for line in log.splitlines() if line]


def fields(line):
    """awk 默认按空白拆列"""
    return line.split()


# 列号提醒：方括号时间 `[10/May/2024:10:00:01 +0800]` 含空格，
# 在 awk 里被拆成 $4 / $5 两列，所以状态码在 $9（Python 下标 8）、路径在 $7（下标 6）。
print("例1（awk 等价）: 提取 5xx 请求的 IP、时间、状态码")
for line in LINES:
    f = fields(line)
    if int(f[8]) >= 500:                      # Python 下标 8 = awk $9 = 状态码
        print(f"  {f[0]:<16} {f[3]:<30} {f[8]}")

print("例2（sed 等价）: 把 UA-1 批量替换成 UA-Edge")
patched = [re.sub(r"UA-1", "UA-Edge", line) for line in LINES]
print("  替换后的前两行:")
for line in patched[:2]:
    print("   ", line)

print("例3（awk 统计等价）: 5xx 接口计数降序（路径 = 下标 6）")
cnt = Counter()
for line in LINES:
    f = fields(line)
    if int(f[8]) >= 500:
        cnt[f[6]] += 1                        # f[6] = "/api/order" 等
for path, n in cnt.most_common():
    print(f"  {n:>2} 次 {path}")


### B6 设计模式三选一：单例 vs 工厂 vs 策略

原笔记 4 已给出各模式的实现，本节聚焦**选型**——面试官更爱问「什么场景用哪个」：

| 维度 | 单例（Singleton） | 工厂（Factory） | 策略（Strategy） |
| --- | --- | --- | --- |
| 解决的核心问题 | 全局唯一实例 + 共享状态 | 解耦「创建」与「使用」 | 运行时**替换算法** |
| 典型场景 | 配置中心、日志器、连接池 | 按配置创建不同实现（渠道/驱动） | 排序/折扣/加密算法可插拔 |
| 变化点 | 无（数量永恒为 1） | 新增产品类型 | 新增算法实现 |
| 代价 | 全局可变状态、难测试 | 工厂本身可能膨胀 | 客户端要能识别算法差异 |
| Python 惯用替代 | 模块级单例 / lru_cache | 字典查找 + 类名 | 传函数 / `sorted(key=...)` |

**什么时候「不」用模式**：单一实现、无变化点、无第二个消费者时，任何模式都是过度设计。
面试加分话术：**「先看变化点，再看消费者数量，最后才套模式」**。

**面试追问 Q&A**

**Q：工厂模式和策略模式长得像，怎么区分？**
A：看**变化点**不同。工厂的变化点是「**要创建哪个类**」（决定对象身份）；策略的变化点是
「**同一职责用哪种算法**」（决定行为）。工厂输出**对象**，策略通常注入到对象内部被调用。
两者可以组合：工厂产出的对象内部再注入策略（下面代码演示）。

**Q：单例为什么难测试？怎么救？**
A：单例把状态怼在全局，测试用例间互相污染（A 用例改了配置，B 用例读到脏值）。修法：
① 单例只存**不可变配置**；② 提供 `_instance = None` 重置钩子；③ 大规模用**依赖注入**——
把对象显式传下去，而不是到处 `Singleton.instance()`。

> ⚠️ **易错点**：① 多线程下单例 `__new__` 有竞态（加锁或模块级初始化）；② 工厂别写成
> 「全能参数化」（一大坨 if 分支比多个小工厂更难维护）；③ 策略模式的经典误用是
> 把「数据不同」当「算法不同」——折扣只是数值不同时用一个带参函数即可，不必建类。


In [ ]:
# B6 实战：工厂（创建支付渠道）+ 策略（折扣算法）组合 + 模块级单例（配置）
# 纯标准库实现

# ---- 策略：折扣算法族 ----
class Discount:
    def apply(self, price):
        raise NotImplementedError


class NoDiscount(Discount):
    def apply(self, price):
        return price


class PercentDiscount(Discount):
    """打折策略：按百分比减价"""

    def __init__(self, percent):
        self.percent = percent

    def apply(self, price):
        return round(price * (1 - self.percent / 100), 2)


class FullReduction(Discount):
    """满减策略：满 threshold 减 amount"""

    def __init__(self, threshold, amount):
        self.threshold = threshold
        self.amount = amount

    def apply(self, price):
        return price - self.amount if price >= self.threshold else price


# ---- 工厂：按名称创建支付渠道 ----
class PayChannel:
    name = ""

    def pay(self, amount):
        raise NotImplementedError


class Alipay(PayChannel):
    name = "支付宝"

    def pay(self, amount):
        return f"[{self.name}] 支付 ¥{amount:.2f}"


class WechatPay(PayChannel):
    name = "微信支付"

    def pay(self, amount):
        return f"[{self.name}] 支付 ¥{amount:.2f}"


class ChannelFactory:
    _channels = {}

    @classmethod
    def register(cls, key):
        def deco(ch):
            cls._channels[key] = ch
            return ch
        return deco

    @classmethod
    def create(cls, key):
        ch = cls._channels.get(key)
        if ch is None:
            raise ValueError(f"未知支付渠道: {key}")
        return ch()


ChannelFactory.register("alipay")(Alipay)
ChannelFactory.register("wechat")(WechatPay)


# ---- 单例：模块级实例（Python 惯用法，天然唯一）----
class Config:
    def __init__(self):
        self.env = "prod"
        self.currency = "CNY"


config = Config()                 # 模块级实例 = 单例（导入一次执行一次）

# ---- 组合使用 ----
print(f"1) 单例配置: env={config.env}, currency={config.currency}")
order_total = 128.0
strategies = {
    "无折扣": NoDiscount(),
    "8 折": PercentDiscount(20),
    "满 100 减 20": FullReduction(100, 20),
}
for name, s in strategies.items():
    final = s.apply(order_total)
    channel = ChannelFactory.create("alipay")
    print(f"2) 策略[{name}] -> {channel.pay(final)}")

# 3) 未知渠道的兜底
try:
    ChannelFactory.create("cash")
except ValueError as e:
    print(f"3) 工厂兜底: {e}")


### B7 性能基准的正确姿势：timeit 的坑与严谨方法

原笔记 5.1 用 timeit 对比过写法，面试追问的往往是**「这个对比可信吗？」**：timeit 只是工具，
**用错姿势结果就失真**。五个经典坑：

1. **单次计时**受系统调度噪音影响 → 用 `repeat` 多次，取 **min（最优）或 median（典型）**；
2. **`number` 大小**：太小测不准（计时器分辨率），太大触发 GC/缓存边界；
3. **对比里混入无关开销**：`map(lambda...)` 慢在**每次迭代调用 Python 函数**，不是 map 本身；
4. **没算内存**：时间快但峰值内存高一样不合格（用 tracemalloc 量）；
5. **没关 GC**：大量临时对象触发 gc 会让数字漂移（benchmark 常用 `gc.disable()`）。

**面试追问 Q&A**

**Q：为什么列表推导通常最快？**
A：推导式编译为专有字节码（`LIST_APPEND` 快速通道），循环在 **C 层**执行 append，避免了
`for` 循环每次 append 的全套字节码求值；`map + lambda` 每次迭代都要调用一次**纯 Python
函数**（调用开销最大）。结论要实测——不同 Python 版本可能微调，面试的标准表述是
「推导式 ≥ for ≥ map+lambda（典型情况）」。

**Q：听到「某写法快 2 倍」，你怎么验证？**
A：① 自己 `timeit.repeat(repeat=7)` 取 min/median 复测；② 检查被比较的两个写法**是否
语义等价**（含不含建列表、异常路径）；③ 看**内存峰值**（tracemalloc）与**可读性**——
工程里「够快 + 好读」优先，极致快只留给热点路径。

> ⚠️ **易错点**：① 别在小数据上比大数据结论（缓存友好性会反转）；② 别把构造开销算进循环体
> （`range` 放外/放内差异）；③ 报告结论要注明**环境**（Python 版本、数据规模、repeat 次数），
> 否则不可复现——面试答「我加了 repeat 取 median」就是专业信号。


In [ ]:
# B7 实战：严谨对比「列表推导 / for 循环 / map+lambda」，并测量峰值内存
import gc
import statistics
import timeit
import tracemalloc

N = 1000          # 数据规模
NUMBER = 2000     # 每组重复次数
REPEAT = 7        # 组数


def f_listcomp():
    return [x * x for x in range(N)]


def f_for():
    acc = []
    for x in range(N):
        acc.append(x * x)
    return acc


def f_map():
    return list(map(lambda x: x * x, range(N)))


def bench(fn):
    """返回 (median_ms, best_ms)；关 GC 减少噪音"""
    gc.disable()
    try:
        times = timeit.repeat(fn, number=NUMBER, repeat=REPEAT)
    finally:
        gc.enable()
    return statistics.median(times) * 1000, min(times) * 1000


def mem_peak(fn):
    """返回函数执行期间的分配峰值（MB）"""
    gc.collect()
    tracemalloc.start()
    fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024 / 1024


print(f"{'实现':<12}{'median(ms)':>12}{'min(ms)':>10}{'峰值内存(MB)':>14}")
results = []
for name, fn in [("列表推导", f_listcomp), ("for 循环", f_for), ("map+lambda", f_map)]:
    med, best = bench(fn)
    mem = mem_peak(fn)
    results.append((name, med, best, mem))
    print(f"{name:<12}{med:>10.3f}{best:>10.3f}{mem:>13.2f}")

results.sort(key=lambda r: r[1])
print(f"\n结论: 本机最快是「{results[0][0]}」。结论受 Python 版本/数据规模影响，"
      f"关键是用 repeat+median 的标准姿势让对比可信。")


### B8 Docker 实战：完整 Dockerfile + 构建运行

一个标准 Python 项目的 Dockerfile（单阶段 + 依赖层缓存优化）：

```dockerfile
# 1. 基镜像：固定 tag，避免漂移
FROM python:3.11-slim

# 2. 防交互卡住 + 设置 Python 环境
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1

# 3. 先拷依赖清单再装依赖 —— 利用层缓存（依赖不变则复用缓存层）
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

# 4. 再拷业务代码（代码变动不触发上一步缓存失效）
COPY . .

# 5. 非 root 运行（安全基线）
RUN useradd -m appuser && chown -R appuser:appuser /app
USER appuser

# 6. 暴露端口 + 启动命令（CMD 可被 docker run 覆盖）
EXPOSE 8000
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:8000", "app:app"]
```

```bash
# 构建（-t 打 tag；-f 指定文件；. 是构建上下文）
docker build -t myapp:1.0 -f Dockerfile .
# 运行（-d 后台；-p 宿主机:容器；--name 起名）
docker run -d --name myapp -p 8000:8000 myapp:1.0
# 常用排查
docker ps                      # 容器是否活着
docker logs -f myapp           # 看日志
docker exec -it myapp bash     # 进容器调试
docker rm -f myapp             # 停并删
```

**CMD vs ENTRYPOINT**：CMD 提供**默认参数**（可被 `docker run` 的命令覆盖）；ENTRYPOINT
是**固定入口**（`docker run` 后面的参数拼到它后面）。JSON 数组（`["gunicorn", ...]`）是
**exec 形式**（推荐，PID 1 正确）；`CMD gunicorn ...` 是 shell 形式（走 `/bin/sh -c`，
信号处理有坑）。

**面试追问 Q&A**

**Q：为什么先 COPY requirements.txt 再 COPY 全部代码？**
A：Docker 的**层缓存**按指令逐层判失效：指令不变 + 上下文文件不变才复用缓存层。
把「不常变」的依赖安装放在「常变」的业务代码之前，改代码时跳过依赖层（最耗时一层），
构建秒级完成；反过来写则每次改代码都重装依赖。

**Q：为什么「容器内 PID 1 必须是 exec 形式」？**
A：shell 形式会让 PID 1 变成 `/bin/sh`，它不转发 SIGTERM，`docker stop` 时进程被强杀、
优雅停机失效。exec 形式 PID 1 才是 gunicorn/nginx master，能正确收到信号。
需要额外 init 时用 `--init`（tini）。

> ⚠️ **易错点**：① 忘写 `.dockerignore`——把 `.git`、`__pycache__`、`.venv` 打进构建上下文
> 会拖慢构建甚至误打包秘钥；② 基镜像 tag 用 `3.11-slim` 而非 `latest`（漂移导致不可复现）；
> ③ 别把容器当数据库存数据——有状态数据用卷挂载；④ `EXPOSE` 只是文档，端口映射靠 `-p`。


In [ ]:
# B8 实战：解析上面这份 Dockerfile，输出每条指令的作用与构建层级
dockerfile = """\
FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
RUN useradd -m appuser && chown -R appuser:appuser /app
USER appuser
EXPOSE 8000
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:8000", "app:app"]
"""

meaning = {
    "FROM": "基镜像（决定运行环境），后续指令都基于它",
    "ENV": "环境变量，构建期与运行期都生效",
    "WORKDIR": "切换工作目录（等价于 cd + 自动 mkdir）",
    "COPY": "从构建上下文复制文件进镜像",
    "RUN": "构建期执行命令（apt/pip/编译），产生新层",
    "USER": "切换运行用户（安全基线：非 root）",
    "EXPOSE": "声明容器监听端口（仅文档，映射靠 -p）",
    "CMD": "启动默认命令（可被 docker run 后面的命令覆盖）",
    "ENTRYPOINT": "固定入口命令（不可覆盖，参数拼接在后面）",
}

print("== Dockerfile 逐条解析 ==")
for lineno, line in enumerate(dockerfile.splitlines(), start=1):
    line = line.strip()
    if not line:
        continue
    ins, _, rest = line.partition(" ")
    ins_name = ins.upper()
    comment = meaning.get(ins_name, "（自定义指令）")
    print(f"{lineno:>2}. {ins_name:<10} {rest:<52} # {comment}")

dockerignore = """\
.git
__pycache__/
*.pyc
*.md
.venv/
"""
print("\n== .dockerignore 示例：防泄漏 + 提速 ==")
for item in dockerignore.splitlines():
    if item.strip():
        print("  -", item)

print("\n== 层缓存推演（为什么先拷 requirements 再拷代码）==")
print("  步骤A: COPY requirements.txt + RUN pip install -> 2 层")
print("  步骤B: COPY . .                                  -> 1 层")
print("  只改 app.py：A 层缓存命中（秒级），仅 B 层重建")
print("  只改 requirements.txt：A、B 两层全部重建（最慢路径）")


### B9 代码审查 Checklist（Review 真实 / 面试通用）

一场 15 分钟的 review 按「正确性 → 安全 → 可维护性 → 性能」四层扫，效率最高：

**① 正确性（最优先）**
- [ ] 边界条件：空输入 / 恰好 1 条 / 超大输入 / 并发一致性
- [ ] 资源泄漏：文件/连接是否 finally 关闭；异常是否被 bare except 静默吞掉
- [ ] Python 陷阱：可变默认参数、闭包延迟绑定、深浅拷贝
- [ ] 时区/编码：`strftime` 本地时区、文件读写 encoding 是否显式

**② 安全**
- [ ] 注入：SQL 拼接、shell 拼接、eval/exec 外部输入
- [ ] 敏感信息：日志/异常里有没有打印 token/密码；`.gitignore` 是否覆盖 `.env`
- [ ] 权限：越权校验在服务端而非前端；用户路径是否归一化

**③ 可维护性**
- [ ] 命名语义化（`fetch_data` 而非 `do_stuff`；布尔变量用 `is_`/`has_` 前缀）
- [ ] 函数单一职责（> 40 行或缩进 > 3 层 → 提醒拆分）
- [ ] 重复代码三处以上 → 抽公共函数；魔法数字 → 常量
- [ ] 新增逻辑是否有对应单测，异常分支是否覆盖

**④ 性能**
- [ ] N+1 查询 / 循环内做 IO / 循环内重复计算不变量
- [ ] 大数据量用生成器/流式；热点路径有 profile 数据支撑
- [ ] 缓存策略：有无缓存、TTL、失效条件

**面试追问 Q&A**

**Q：Review 里发现低级错误但作者不认同，怎么沟通？**
A：**对事不对人**，用证据说话：先复述「我理解的逻辑」，再给**可复现的最小示例**
（一段能跑出异常结果的代码），最后给建议写法。把「你写错了」换成「如果输入是 X，
这段代码的结果是 Y，与预期 Z 不一致，是不是边界没覆盖？」——面试官考的就是这种协作话术。

**Q：Review 和工具（linter/mypy）的关系？**
A：分层防线：**格式与简单规则交给工具**（linter、pre-commit 钩子），**人审聚焦工具看不出
的语义问题**（设计、边界、并发、安全）。Review 里看到「能自动检查的」，应反馈给作者加
CI 规则而不是每次人工提醒——这是「解决根因」与「治标」的区别。

> ⚠️ **易错点**：① 只点赞不挑刺 = 无效 review（走形式最伤团队信任）；但**堆砌无价值 nit**
> 也伤士气——把意见分为 Must fix / Should fix / Nit 三档；② 别在 merge 后 review
> （成本最高）——**小而频繁**的 PR 是好的 review 的前提；③ 审查的最重要产出是
> **作者与审查者达成共识**，不是找茬数量。


In [ ]:
# B9 实战：用 AST 写一个「审查小助手」，自动挑出常见问题（演示审查自动化思路）
import ast

BAD_CODE = """\
import time


def fetch_user(user_id, cache={}):
    # 问题1: 可变默认参数（cache 会被所有调用共享累积）
    data = query_db(user_id)
    if data.status == 200:
        _do_work(data)
    else:
        print(data.error)
    return data


try:
    result = fetch_user(1)
except:
    # 问题2: bare except 吞掉一切异常（包括 KeyboardInterrupt/SystemExit）
    pass

if result is 1:
    # 问题3: is 比较整数（依赖小整数池/常量去重，不可靠）
    pass


def query_db(uid):
    return type("R", (), {"status": 200, "error": "DB down"})()


def _do_work(data):
    for i in range(3):
        time.sleep(0.01)   # 问题4（示意）: 循环内阻塞调用
"""


def review(source):
    """用 AST 检查常见问题，返回问题清单（按严重级别标注）"""
    issues = []
    tree = ast.parse(source)

    for node in ast.walk(tree):
        # 1) 可变默认参数
        if isinstance(node, ast.FunctionDef):
            for d in node.args.defaults:
                if isinstance(d, (ast.List, ast.Dict, ast.Set)):
                    issues.append(
                        f"[Must] 第{node.lineno}行 {node.name}: 可变默认参数"
                        f"（每次调用共享同一对象）")

        # 2) bare except
        if isinstance(node, ast.ExceptHandler) and node.type is None:
            issues.append(f"[Must] 第{node.lineno}行: bare except 会吞掉 BaseException")

        # 3) 用 is 比较字面量（None/True/False 除外）
        if isinstance(node, ast.Compare):
            for op, right in zip(node.ops, node.comparators):
                if (isinstance(op, ast.Is)
                        and isinstance(right, ast.Constant)
                        and not isinstance(right.value, (type(None), bool))):
                    issues.append(f"[Should] 第{node.lineno}行: `is` 用于字面量比较（应改用 ==）")

        # 4) 循环体里的阻塞调用（示意性检查）
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                and node.func.attr == "sleep"):
            issues.append(f"[Nit] 第{node.lineno}行: time.sleep 阻塞调用（循环内要合并/异步化）")

    return issues


print("== 被审查代码中发现的问题 ==")
for issue in review(BAD_CODE):
    print(" ", issue)

tree = ast.parse(BAD_CODE)
n_funcs = sum(isinstance(n, ast.FunctionDef) for n in ast.walk(tree))
n_lines = len(BAD_CODE.splitlines())
print(f"\n== 规模统计 ==: {n_lines} 行, {n_funcs} 个函数（单函数 > 40 行值得拆分）")
print("（真实 review 中，能被 AST 自动查出的规则应沉淀进 CI/linter，人工聚焦语义问题）")